## Dataset Preprocessing for SETRec (Step 1)

- This notebook contains the pre-processing step 1 for producing experimental datasets. The step 1 obtain the training/validation/testing interactions and output necessary files accordingly.
Note that this notebook is a post-processed one, as we delete some useless intermediate printing.

- The dataset pre-processing also includes step 2, where we obtain the meta information and save the textual meta information into "combine_tdcb_maps.npy" file. (Please refer to dataset_preprocess_step2.ipynb).

- This notebook is an example of the "Toys" dataset.

- notes: we use amazon2014 dataset.

In [3]:
import array
import time
import gzip
import copy
import math
import numpy as np
import pandas as pd 
import pdb
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pandas.core.frame import DataFrame

import torch

import random
random_seed = 1
torch.manual_seed(random_seed) # cpu
# torch.cuda.manual_se/ed(random_seed) #gpu
np.random.seed(random_seed) #numpy
random.seed(random_seed) #random and transforms
torch.backends.cudnn.deterministic=True # cudnn

In [4]:
import json

In [ ]:
def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        yield json.loads(l)

reviews_path = '/your/path/to/raw/data/reviews_Toys_and_Games_5.json.gz'

# ratings = pd.read_csv(ratings_path,sep='\t')
reviews = parse(reviews_path)

In [6]:
def select_kcore(_interaction_dict, K=10, verbose=False):
    interaction_dict = copy.deepcopy(_interaction_dict)
    flag = 0
    while flag==0:
        item_cnt_dict = {}
        item_drop_dict = {}
        # create item_drop_dict, item_cnt_dict
        for user_id in interaction_dict:
            for item_id in interaction_dict[user_id]:
                item_cnt_dict[item_id] = item_cnt_dict.get(item_id, 0) + 1
                item_drop_dict[item_id] = 0
                
        #print('user num:',len(interaction_dict))
        assert len(item_drop_dict)==len(item_cnt_dict)

        # delete items < K
        del_iid_list = []
        for i_id in item_cnt_dict:
            if item_cnt_dict[i_id] < K:
                del_iid_list.append(i_id)

        for i_id in del_iid_list:
            item_drop_dict[i_id] = 1
        for u_id in interaction_dict:
            del_id_list = []
            for i_id in interaction_dict[u_id]:
                if item_drop_dict[i_id]:
                    del_id_list.append(i_id)
            for del_id in del_id_list:
                del interaction_dict[u_id][del_id]

        item_drop_num = 0
        for i_id in item_drop_dict:
            item_drop_num += item_drop_dict[i_id]
        item_num = len(item_drop_dict) - item_drop_num
#         print(f'item num after item-{K}core:',item_num)

        new_item_cnt = {}
        min_cnt=9999
        for u_id in interaction_dict:
            min_cnt = min(min_cnt, len(interaction_dict[u_id]))
            for i_id in interaction_dict[u_id]:
                new_item_cnt[i_id] = new_item_cnt.get(i_id, 0) + 1
            
        min_cnt_item = 9999
        for i_id in new_item_cnt:
            min_cnt_item = min(min_cnt_item, new_item_cnt[i_id])
            
        if verbose:
            print('min user interaction:',min_cnt)
            print('min item num:',min_cnt_item)
            
        if min_cnt>=K and min_cnt_item>=K:
            return interaction_dict, len(interaction_dict), item_num
        
        # delete users interactions<K
        del_uid_list = []
        for u_id in interaction_dict:
            if len(interaction_dict[u_id])<K:
                del_uid_list.append(u_id)
        for u_id in del_uid_list:
            del interaction_dict[u_id]
        
        # count min user-interaction and item appearance
        new_item_cnt = {}
        min_cnt=9999
        for u_id in interaction_dict:
            min_cnt = min(min_cnt, len(interaction_dict[u_id]))
            for i_id in interaction_dict[u_id]:
                new_item_cnt[i_id] = new_item_cnt.get(i_id, 0) + 1
                 
        min_cnt_item = 9999
        for i_id in new_item_cnt:
            min_cnt_item = min(min_cnt_item, new_item_cnt[i_id])

        if verbose:
            print('min user interaction:',min_cnt)
            print('min item num:',min_cnt_item)
            
        if min_cnt>=K and min_cnt_item>=K:
            return interaction_dict, len(interaction_dict), item_num

In [7]:
for review in reviews:
    print(review)
    print(type(review))
    break

{'reviewerID': 'A1VXOAVRGKGEAK', 'asin': '0439893577', 'reviewerName': 'Angie', 'helpful': [0, 0], 'reviewText': 'I like the item pricing. My granddaughter wanted to mark on it but I wanted it just for the letters.', 'overall': 5.0, 'summary': 'Magnetic board', 'unixReviewTime': 1390953600, 'reviewTime': '01 29, 2014'}
<class 'dict'>


In [8]:
interaction_dict = {}
cnt=0
interaction_num = 0
raw_item = set()
for review in reviews:
    try:
        u_id, i_id, rating, time = review['reviewerID'], review['asin'], review['overall'], review['unixReviewTime']
        if u_id not in interaction_dict:
            interaction_dict[u_id] = {}
        interaction_dict[u_id][i_id] = time
        interaction_num += 1
        raw_item.add(i_id)
    except:
        print(review)
        cnt+=1
print('raw user num:',len(interaction_dict))
print('raw item num:', len(raw_item))
print('total interaction num:', interaction_num)
print(cnt)

raw user num: 19412
raw item num: 11924
total interaction num: 167596
0


In [9]:
import copy

In [46]:
# sort each user's interaction by timestamp
interaction_dict_new = copy.deepcopy(interaction_dict)
for u_id in interaction_dict_new:
    interaction_dict_new[u_id] = dict(sorted(interaction_dict_new[u_id].items(),key=lambda item:item[1]))

### 1. k-core selection 

In [47]:
# k-core selection
interaction_dict_new, user_num, item_num = select_kcore(interaction_dict_new,0)
print('after 0 core...')
print('user num:',user_num)
print('item num:',item_num)

after 0 core...
user num: 19412
item num: 11924


In [48]:
len(interaction_dict_new)

19412

### following split process

In [53]:
time_list = []
for u_id in interaction_dict_new:
    for i_id in interaction_dict_new[u_id]:
        time_list.append(interaction_dict_new[u_id][i_id])
time_list = sorted(time_list)

In [54]:
len(time_list)

167596

In [55]:
training_old_dict, validation_old_dict, testing_old_dict = {}, {}, {}

# 0.134&2
split_ratio = 0.17
test_num = int(len(time_list)*split_ratio)
split_time1 = time_list[-test_num]
split_time2 = time_list[-math.ceil(1.8*test_num)]
print('*-------')
print(test_num, math.ceil(2*test_num))
print(split_time1, split_time2)
print(time_list[0], time_list[-1])

for u_id in interaction_dict_new:
    training_old_dict[u_id] = []
    validation_old_dict[u_id] = []
    testing_old_dict[u_id] = []
    for i_id, time in interaction_dict_new[u_id].items():
        if time < split_time2:
            training_old_dict[u_id].append(i_id)
        elif time < split_time1:
            validation_old_dict[u_id].append(i_id)
        else:
            testing_old_dict[u_id].append(i_id)

cut_user = 0

for u_id in interaction_dict_new:
    if len(training_old_dict[u_id]) >= 2: # remove users with less than 2 interactions in training dict
        continue
    else:
        cut_user += 1
        del training_old_dict[u_id]
        del validation_old_dict[u_id]
        del testing_old_dict[u_id]

print(cut_user)

*-------
28491 56982
1390262400 1384905600
964742400 1406073600
3777


In [56]:
# use list to store user/item for map generation -> for reproducibility

def get_unique_sorted_elements(lst):
    return sorted(list(set(lst)))

user_set = []
item_set = []
for u_id in training_old_dict:
    user_set.append(u_id)
    for i_id in training_old_dict[u_id]:
        item_set.append(i_id)
    for i_id in validation_old_dict[u_id]:
        item_set.append(i_id)
    for i_id in testing_old_dict[u_id]:
        item_set.append(i_id)

item_set = get_unique_sorted_elements(item_set)
            
import random
random.seed(2023)
random.shuffle(item_set)

user_map = {old_id:new_id for new_id, old_id in enumerate(user_set)}
item_map = {old_id:new_id for new_id, old_id in enumerate(item_set)}

user_map = dict(sorted(user_map.items(),key=lambda item:item[1]))
item_map = dict(sorted(item_map.items(),key=lambda item:item[1]))

save_path = './'
np.save(save_path + 'user_map.npy',user_map)
np.save(save_path + 'item_map.npy',item_map)

user_map_reverse = {k:v for v,k in user_map.items()}
item_map_reverse = {k:v for v,k in item_map.items()}
np.save(save_path + 'user_map_reverse.npy',user_map_reverse)
np.save(save_path + 'item_map_reverse.npy',item_map_reverse)

print('user num:', len(user_set))
print('item num:', len(item_set))

user num: 15635
item num: 11908


In [57]:
warm_item_set, cold_item_set = set(), set()
valid_warm_item_set, valid_cold_item_set, test_warm_item_set, test_cold_item_set = set(), set(), set(), set()
training_interaction, validation_warm_interaction, validation_cold_interaction, testing_warm_interaction, testing_cold_interaction = 0, 0, 0, 0, 0
training_user, validation_warm_user, validation_cold_user, testing_warm_user, testing_cold_user = 0, 0, 0, 0, 0
validation_overlap_user, testing_overlap_user = 0, 0
for u_id in training_old_dict:
    training_user += 1
    for i_id in training_old_dict[u_id]:
        warm_item_set.add(item_map[i_id])
        training_interaction += 1
for u_id in validation_old_dict:
    flag_w, flag_c = 0, 0
    for i_id in validation_old_dict[u_id]:
        if item_map[i_id] in warm_item_set:
            valid_warm_item_set.add(item_map[i_id])
            validation_warm_interaction += 1
            flag_w = 1
        else:
            cold_item_set.add(item_map[i_id])
            valid_cold_item_set.add(item_map[i_id])
            validation_cold_interaction += 1
            flag_c = 1
    if flag_w == 1:
        validation_warm_user += 1
    if flag_c == 1:
        validation_cold_user += 1
    if flag_w == 1 and flag_c == 1:
        validation_overlap_user += 1
for u_id in testing_old_dict:
    flag_w, flag_c = 0, 0
    for i_id in testing_old_dict[u_id]:
        if item_map[i_id] in warm_item_set:
            test_warm_item_set.add(item_map[i_id])
            testing_warm_interaction += 1
            flag_w = 1
        else:
            cold_item_set.add(item_map[i_id])
            test_cold_item_set.add(item_map[i_id])
            testing_cold_interaction += 1
            flag_c = 1
    if flag_w == 1:
        testing_warm_user += 1
    if flag_c == 1:
        testing_cold_user += 1
    if flag_w == 1 and flag_c == 1:
        testing_overlap_user += 1

test_user_num = 0
for u_id in testing_old_dict:
    if len(testing_old_dict[u_id]):
        test_user_num += 1
        
tot_interaction = training_interaction + validation_warm_interaction + validation_cold_interaction + testing_warm_interaction + testing_cold_interaction

print('warm item num:', len(warm_item_set))
print('cold item num:', len(cold_item_set))
print('valid warm item num:', len(valid_warm_item_set))
print('valid cold item num:', len(valid_cold_item_set))
print('test warm item num:', len(test_warm_item_set))
print('test cold item num:', len(test_cold_item_set))
print('----------------')
print('training interaction num:', training_interaction)
print('validation warm interaction num:', validation_warm_interaction)
print('validation cold interaction num:', validation_cold_interaction)
print('testing warm interaction num:', testing_warm_interaction)
print('testing cold interaction num:', testing_cold_interaction)
vdt = round((validation_warm_interaction+validation_cold_interaction)/(testing_warm_interaction+testing_cold_interaction), 1)
trdv = round(training_interaction/(validation_warm_interaction+validation_cold_interaction), 1)
trdt = trdv*vdt
print(f"#inter-train:valid:test = {trdt}:{vdt}:1")
print('----------------')
print('training warm user num:', training_user)
print('validation warm user num:', validation_warm_user)
print('validation cold user num:', validation_cold_user)
print('testing warm user num:', testing_warm_user)
print('testing cold user num:', testing_cold_user)
print('----------------')
print('user num:', len(user_set))
print('item num:', len(item_set))
print('interaction:', tot_interaction)
print('density:', tot_interaction/(len(user_set)*len(item_set)))
print('----------------')
print('validation overlap user num:', validation_overlap_user)
print('testing overlap user num:', testing_overlap_user)
print('----------------')
print('test user num:', test_user_num)

warm item num: 11285
cold item num: 623
valid warm item num: 4929
valid cold item num: 330
test warm item num: 5111
test cold item num: 572
----------------
training interaction num: 114983
validation warm interaction num: 11686
validation cold interaction num: 1054
testing warm interaction num: 10541
testing cold interaction num: 4235
#inter-train:valid:test = 8.1:0.9:1
----------------
training warm user num: 15635
validation warm user num: 4715
validation cold user num: 856
testing warm user num: 4489
testing cold user num: 2053
----------------
user num: 15635
item num: 11908
interaction: 142499
density: 0.000765376467424973
----------------
validation overlap user num: 610
testing overlap user num: 983
----------------
test user num: 5559


In [58]:
training_dict, validation_dict, testing_dict = {}, {}, {}
training_list, validation_list, testing_list = [], [], []
validation_warm_dict, validation_cold_dict, testing_warm_dict, testing_cold_dict = {}, {}, {}, {}

for u_id in training_old_dict:
    training_dict[user_map[u_id]] = []
    for i_id in training_old_dict[u_id]:
        training_dict[user_map[u_id]].append(item_map[i_id])
        training_list.append([user_map[u_id], item_map[i_id]])
for u_id in validation_old_dict:
    validation_dict[user_map[u_id]] = []
    validation_warm_dict[user_map[u_id]] = []
    validation_cold_dict[user_map[u_id]] = []
    for i_id in validation_old_dict[u_id]:
        validation_dict[user_map[u_id]].append(item_map[i_id])
        validation_list.append([user_map[u_id], item_map[i_id]])
        if item_map[i_id] in warm_item_set:
            validation_warm_dict[user_map[u_id]].append(item_map[i_id])
        else:
            validation_cold_dict[user_map[u_id]].append(item_map[i_id])
for u_id in testing_old_dict:
    testing_dict[user_map[u_id]] = []
    testing_warm_dict[user_map[u_id]] = []
    testing_cold_dict[user_map[u_id]] = []
    for i_id in testing_old_dict[u_id]:
        testing_dict[user_map[u_id]].append(item_map[i_id])
        testing_list.append([user_map[u_id], item_map[i_id]])
        if item_map[i_id] in warm_item_set:
            testing_warm_dict[user_map[u_id]].append(item_map[i_id])
        else:
            testing_cold_dict[user_map[u_id]].append(item_map[i_id])

In [59]:
valid_warm_set, valid_cold_set, test_warm_set, test_cold_set = set(), set(), set(), set()
for u_id in validation_warm_dict:
    for i_id in validation_warm_dict[u_id]:
        valid_warm_set.add(i_id)
for u_id in validation_cold_dict:
    for i_id in validation_cold_dict[u_id]:
        valid_cold_set.add(i_id)
for u_id in testing_warm_dict:
    for i_id in testing_warm_dict[u_id]:
        test_warm_set.add(i_id)
for u_id in testing_cold_dict:
    for i_id in testing_cold_dict[u_id]:
        test_cold_set.add(i_id)

print(len(valid_warm_set))
print(len(valid_cold_set))
print(len(test_warm_set))
print(len(test_cold_set))

print('valid warm item num:', len(valid_warm_item_set))
print('valid cold item num:', len(valid_cold_item_set))
print('test warm item num:', len(test_warm_item_set))
print('test cold item num:', len(test_cold_item_set))

4929
330
5111
572
valid warm item num: 4929
valid cold item num: 330
test warm item num: 5111
test cold item num: 572


In [60]:
np.save(save_path + 'training_list.npy', np.array(training_list))
np.save(save_path + 'training_dict.npy', training_dict)
np.save(save_path + 'validation_dict.npy', validation_dict)
np.save(save_path + 'testing_dict.npy', testing_dict)
np.save(save_path + 'validation_warm_dict.npy', validation_warm_dict)
np.save(save_path + 'validation_cold_dict.npy', validation_cold_dict)
np.save(save_path + 'testing_warm_dict.npy', testing_warm_dict)
np.save(save_path + 'testing_cold_dict.npy', testing_cold_dict)
np.save(save_path + 'warm_item.npy', warm_item_set)
np.save(save_path + 'cold_item.npy', cold_item_set)

### overlap check

In [61]:
def overlap(dict0,dict1,dict2):
    
    count1 = {key:0 for key in dict0}
    count2 = {key:0 for key in dict0}
    res = {key:set() for key in dict0}

    for u_id,items in dict1.items():
        count1[u_id] = len(items)
    for u_id,items in dict2.items():
        count2[u_id] = len(items)
    for u_id in res:
        try:
            for item in dict1[u_id]:
                res[u_id].add(item)
        except:
            pass
        try:
            for item in dict2[u_id]:
                res[u_id].add(item)
        except:
            pass

    cnt=0
    interaction_cnt=0
    for u_id in res:
        if len(res[u_id])!= count1[u_id] + count2[u_id]:
            cnt = cnt + count1[u_id] + count2[u_id] - len(res[u_id])
        interaction_cnt += len(res[u_id])
    ratio = cnt/interaction_cnt
    return cnt,interaction_cnt,ratio

In [62]:
time_tr  = np.load('training_dict.npy',allow_pickle=True).item()
time_val = np.load('validation_dict.npy',allow_pickle=True).item()
time_tst = np.load('testing_dict.npy',allow_pickle=True).item()
print('tr - val:',overlap(time_tr, time_tr, time_val ))
print('tr - tst:',overlap(time_tr, time_tr, time_tst ))
print('val - tst:',overlap(time_tr, time_tst, time_val ))

tr - val: (0, 127723, 0.0)
tr - tst: (0, 129759, 0.0)
val - tst: (0, 27516, 0.0)


### testing statistics

In [63]:
cnt = 0
for u_id in time_tst:
    if time_tst[u_id]:
        cnt +=1
print(f"testing user num: {cnt}")

testing user num: 5559
